In [1]:
import urllib.request
import zipfile
import os

url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
zip_path = "../data/ml-100k.zip"

os.makedirs("../data", exist_ok=True)
urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("../data")

print("İndirme ve çıkarma tamamlandı.")
print(os.listdir("../data/ml-100k"))

İndirme ve çıkarma tamamlandı.
['allbut.pl', 'mku.sh', 'README', 'u.data', 'u.genre', 'u.info', 'u.item', 'u.occupation', 'u.user', 'u1.base', 'u1.test', 'u2.base', 'u2.test', 'u3.base', 'u3.test', 'u4.base', 'u4.test', 'u5.base', 'u5.test', 'ua.base', 'ua.test', 'ub.base', 'ub.test']


In [3]:
import pandas as pd

# Puanlamalar
ratings = pd.read_csv(
    '../data/ml-100k/u.data',
    sep='\t',
    names=['userId', 'movieId', 'rating', 'timestamp']
)

# Film bilgileri
movie_cols = ['movieId', 'title', 'release_date', 'video_release_date', 'imdb_url'] + \
             ['genre_' + str(i) for i in range(19)]

movies = pd.read_csv(
    '../data/ml-100k/u.item',
    sep='|',
    names=movie_cols,
    encoding='latin-1'
)

print("Ratings shape:", ratings.shape)
print("Movies shape:", movies.shape)
ratings.head()

Ratings shape: (100000, 4)
Movies shape: (1682, 24)


,userId,movieId,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [4]:
movies[['movieId', 'title'] + [f'genre_{i}' for i in range(19)]].head()

,movieId,title,genre_0,genre_1,genre_2,genre_3,genre_4,genre_5,genre_6,genre_7,...,genre_9,genre_10,genre_11,genre_12,genre_13,genre_14,genre_15,genre_16,genre_17,genre_18
0,1,Toy Story (1995),0,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0


In [5]:
genre_names = pd.read_csv('../data/ml-100k/u.genre', sep='|', header=None, names=['genre', 'genre_id'])
genre_names

,genre,genre_id
0,unknown,0
1,Action,1
2,Adventure,2
3,Animation,3
4,Children's,4
5,Comedy,5
6,Crime,6
7,Documentary,7
8,Drama,8
9,Fantasy,9


In [6]:
genre_list = genre_names['genre'].tolist()  # 19 tür adı, sırayla
genre_cols = [f'genre_{i}' for i in range(19)]

def get_genres(row):
    return '|'.join([genre_list[i] for i in range(19) if row[genre_cols[i]] == 1])

movies['genres'] = movies.apply(get_genres, axis=1)

movies[['movieId', 'title', 'genres']].head(10)

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,GoldenEye (1995),Action|Adventure|Thriller
2,3,Four Rooms (1995),Thriller
3,4,Get Shorty (1995),Action|Comedy|Drama
4,5,Copycat (1995),Crime|Drama|Thriller
5,6,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,Drama
6,7,Twelve Monkeys (1995),Drama|Sci-Fi
7,8,Babe (1995),Children's|Comedy|Drama
8,9,Dead Man Walking (1995),Drama
9,10,Richard III (1995),Drama|War


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# '|' karakterini boşluğa çevirip TF-IDF'in kelime kelime ayırmasını sağlıyoruz
movies['genres_str'] = movies['genres'].str.replace('|', ' ', regex=False)

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['genres_str'])

print("TF-IDF matrix boyutu:", tfidf_matrix.shape)

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("Cosine similarity matrix boyutu:", cosine_sim.shape)

TF-IDF matrix boyutu: (1682, 21)
Cosine similarity matrix boyutu: (1682, 1682)


In [8]:
def get_recommendations(title, n=5):
    idx = movies[movies['title'] == title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]  # ilk sıradaki filmin kendisi, onu atla
    movie_indices = [i[0] for i in sim_scores]
    return movies.iloc[movie_indices][['title', 'genres']]

get_recommendations('Toy Story (1995)')

,title,genres
421,Aladdin and the King of Thieves (1996),Animation|Children's|Comedy
101,"Aristocats, The (1970)",Animation|Children's
403,Pinocchio (1940),Animation|Children's
624,"Sword in the Stone, The (1963)",Animation|Children's
945,"Fox and the Hound, The (1981)",Animation|Children's


In [9]:
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

print("Matris boyutu:", user_movie_matrix.shape)
user_movie_matrix.head()

Matris boyutu: (943, 1682)


movieId,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
userId,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# NaN'ları 0 ile dolduruyoruz (cosine similarity hesaplaması için gerekli)
movie_user_matrix = user_movie_matrix.fillna(0).T  # transpoze: satır=film, sütun=kullanıcı

from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(movie_user_matrix)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=movie_user_matrix.index,
    columns=movie_user_matrix.index
)

print("Item similarity matrix boyutu:", item_similarity_df.shape)
item_similarity_df.head()

Item similarity matrix boyutu: (1682, 1682)


movieId,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.402382,0.330245,0.454938,0.286714,0.116344,0.620979,0.481114,0.496288,0.273935,...,0.035387,0.0,0.000000,0.000000,0.035387,0.0,0.0,0.0,0.047183,0.047183
2,0.402382,1.000000,0.273069,0.502571,0.318836,0.083563,0.383403,0.337002,0.255252,0.171082,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.078299,0.078299
3,0.330245,0.273069,1.000000,0.324866,0.212957,0.106722,0.372921,0.200794,0.273669,0.158104,...,0.000000,0.0,0.000000,0.000000,0.032292,0.0,0.0,0.0,0.000000,0.096875
4,0.454938,0.502571,0.324866,1.000000,0.334239,0.090308,0.489283,0.490236,0.419044,0.252561,...,0.000000,0.0,0.094022,0.094022,0.037609,0.0,0.0,0.0,0.056413,0.075218
5,0.286714,0.318836,0.212957,0.334239,1.000000,0.037299,0.334769,0.259161,0.272448,0.055453,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.094211
